In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.regression.mixed_linear_model import MixedLM

In [2]:
combined_df = pd.read_excel('Combined_Time_Fix_Data.xlsx')
Dataset_Values = combined_df['Dataset'].unique()
columns = combined_df.columns.values
print(combined_df[columns[3]].unique())
combined_df.drop(columns=columns[0], axis=1, inplace=True) # inplace to modify combined_df instead of copy of object
combined_df.info()
# combined_df[combined_df['Dataset']==Dataset_Values[2]].head()

['Chemotaxis_Diffusion' 'Conjugation_Ra' 'SOS_Spontaneous']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 794 entries, 0 to 793
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Condition    794 non-null    object 
 1   Time         600 non-null    float64
 2   Dataset      794 non-null    object 
 3   Log_Time     0 non-null      float64
 4   Conjugation  794 non-null    bool   
 5   R_a          794 non-null    bool   
 6   Chemotaxis   794 non-null    bool   
 7   SOS          794 non-null    bool   
dtypes: bool(4), float64(2), object(2)
memory usage: 28.0+ KB


### Data Cleaning

In [3]:
# Removing Outliers
def remove_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]

# Function to clean data per dataset
def clean_data_by_dataset(df, dataset_column, target_column):
    cleaned_df_list = []
    unique_datasets = df[dataset_column].unique() # Get unique names from 'Dataset' column.
    
    for dataset in unique_datasets:
        dataset_df = df[df[dataset_column] == dataset]
        cleaned_dataset_df = remove_outliers(dataset_df, target_column) # 'Time' is the target column to remove the outliers from.
        cleaned_df_list.append(cleaned_dataset_df)
    
    cleaned_combined_df = pd.concat(cleaned_df_list)
    cleaned_combined_df.reset_index(drop=True, inplace=True)
    return cleaned_combined_df

In [4]:
# Apply the function
combined_df = clean_data_by_dataset(combined_df, 'Dataset', 'Time')

# Check the result
combined_df[combined_df['Dataset']=='SOS_Spontaneous'].info()
combined_df['Log_Time'] = combined_df['Time'].apply(lambda x: np.log(x))
combined_df

<class 'pandas.core.frame.DataFrame'>
Index: 0 entries
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Condition    0 non-null      object 
 1   Time         0 non-null      float64
 2   Dataset      0 non-null      object 
 3   Log_Time     0 non-null      float64
 4   Conjugation  0 non-null      bool   
 5   R_a          0 non-null      bool   
 6   Chemotaxis   0 non-null      bool   
 7   SOS          0 non-null      bool   
dtypes: bool(4), float64(2), object(2)
memory usage: 0.0+ bytes


,Condition,Time,Dataset,Log_Time,Conjugation,R_a,Chemotaxis,SOS
0,Chemotaxis,16634.0,Chemotaxis_Diffusion,9.719204,True,True,True,True
1,Chemotaxis,22472.0,Chemotaxis_Diffusion,10.020025,True,True,True,True
2,Chemotaxis,15586.0,Chemotaxis_Diffusion,9.654128,True,True,True,True
3,Chemotaxis,15735.0,Chemotaxis_Diffusion,9.663643,True,True,True,True
4,Chemotaxis,15363.0,Chemotaxis_Diffusion,9.639717,True,True,True,True
...,...,...,...,...,...,...,...,...
583,No R_a No Conj,7081.0,Conjugation_Ra,8.865170,False,False,True,True
584,No R_a No Conj,7508.0,Conjugation_Ra,8.923724,False,False,True,True
585,No R_a No Conj,7013.0,Conjugation_Ra,8.855521,False,False,True,True
586,No R_a No Conj,4643.0,Conjugation_Ra,8.443116,False,False,True,True


### Conjugation R_a Regression Analysis

In [5]:
# Filter the dataframe to exclude the 'Chemotaxis_Diffusion' dataset
filtered_df = combined_df[combined_df['Dataset'] == 'Conjugation_Ra'].copy()

# Create the new column with opposite boolean values
filtered_df["No_R_a"] = ~filtered_df["R_a"]

# Add interaction term for R_a * Conjugation
filtered_df['R_a_Conj'] = filtered_df['R_a'] & filtered_df['Conjugation']

# Add interaction term for No_R_a * Conjugation
filtered_df['No_R_a_Conj'] = filtered_df['No_R_a'] & filtered_df['Conjugation']

# Display the first few rows to verify the new column
filtered_df[['Condition', 'Time', 'Dataset', 'Log_Time', 'Conjugation', 'R_a', 'R_a_Conj', 'No_R_a_Conj', 'No_R_a']].head()

,Condition,Time,Dataset,Log_Time,Conjugation,R_a,R_a_Conj,No_R_a_Conj,No_R_a
196,Normal Model,3124.0,Conjugation_Ra,8.046870,True,True,True,False,False
197,Normal Model,6104.0,Conjugation_Ra,8.716700,True,True,True,False,False
198,Normal Model,3418.0,Conjugation_Ra,8.136811,True,True,True,False,False
199,Normal Model,4620.0,Conjugation_Ra,8.438150,True,True,True,False,False
200,Normal Model,4282.0,Conjugation_Ra,8.362175,True,True,True,False,False


In [7]:
# Prepare the model formula including the interaction term
formula = 'Log_Time ~ Conjugation + No_R_a'

# Fit the OLS regression model
model_interaction = sm.OLS.from_formula(formula, data=filtered_df).fit()

# Display the summary of the model
model_interaction_summary = model_interaction.summary()
print(model_interaction_summary)

                            OLS Regression Results                            
Dep. Variable:               Log_Time   R-squared:                       0.117
Model:                            OLS   Adj. R-squared:                  0.112
Method:                 Least Squares   F-statistic:                     25.65
Date:                Sun, 21 Jul 2024   Prob (F-statistic):           3.42e-11
Time:                        21:08:21   Log-Likelihood:                -200.46
No. Observations:                 392   AIC:                             406.9
Df Residuals:                     389   BIC:                             418.8
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept               8.5401    

In [12]:
filtered_df['No_R_a_Conj'].info()
# Prepare the model formula including the interaction term
formula = 'Log_Time ~ Conjugation * No_R_a'

# Fit the OLS regression model
model_interaction = sm.OLS.from_formula(formula, data=filtered_df).fit()

# Display the summary of the model
model_interaction_summary = model_interaction.summary()
print(model_interaction_summary)

<class 'pandas.core.series.Series'>
Index: 392 entries, 196 to 587
Series name: No_R_a_Conj
Non-Null Count  Dtype
--------------  -----
392 non-null    bool 
dtypes: bool(1)
memory usage: 3.4 KB
                            OLS Regression Results                            
Dep. Variable:               Log_Time   R-squared:                       0.132
Model:                            OLS   Adj. R-squared:                  0.125
Method:                 Least Squares   F-statistic:                     19.65
Date:                Mon, 22 Jul 2024   Prob (F-statistic):           7.08e-12
Time:                        16:34:56   Log-Likelihood:                -197.03
No. Observations:                 392   AIC:                             402.1
Df Residuals:                     388   BIC:                             417.9
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
               

In [22]:
# Prepare the model formula including the interaction term
formula = 'Log_Time ~ Conjugation + No_R_a_Conj'

# Fit the OLS regression model
model_interaction = sm.OLS.from_formula(formula, data=filtered_df).fit()

# Display the summary of the model
model_interaction_summary = model_interaction.summary()
print(model_interaction_summary)

                            OLS Regression Results                            
Dep. Variable:               Log_Time   R-squared:                       0.132
Model:                            OLS   Adj. R-squared:                  0.127
Method:                 Least Squares   F-statistic:                     29.54
Date:                Mon, 22 Jul 2024   Prob (F-statistic):           1.13e-12
Time:                        17:03:42   Log-Likelihood:                -197.03
No. Observations:                 392   AIC:                             400.1
Df Residuals:                     389   BIC:                             412.0
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept               8.4866    

The interaction term R_a_Conj is statistically significant in the simpler model because this model directly captures the combined effect without the added degrees of freedom, larger standard errors, and potential collinearity of separate terms, thereby reducing the t-values and making it harder to achieve statistical significance.

**Conclusion of Analysis and Comparison to Permutation Test:**
- Conjugation combined with a lack of R_a is the most statistically significant predictor, coef = -0.3226 (p < 0.0001).
- R_a being absent is not significant.
- Conj being present is significant (p < 0.001).
- The permutation test showed practical significance for the interaction effect between the synergystic effect of no R(a) and conjugation.
- Regression results for No_R_a_Conj indicate a significant interaction effect in the model focusing on this term, although the regression model with both conjugation and no R(a) has a larger R-squared indicating a better fit with variance.

### Chemotaxis Diffusion Regression Analysis

In [20]:
# Filter the dataframe to include only the 'Chemotaxis_Diffusion' dataset
chemotaxis_df = combined_df[combined_df['Dataset'] == 'Chemotaxis_Diffusion'].copy()

# Display the first few rows to verify
chemotaxis_df[['Condition', 'Time', 'Dataset', 'Log_Time', 'Chemotaxis']].head()

,Condition,Time,Dataset,Log_Time,Chemotaxis
0,Chemotaxis,16634.0,Chemotaxis_Diffusion,9.719204,True
1,Chemotaxis,22472.0,Chemotaxis_Diffusion,10.020025,True
2,Chemotaxis,15586.0,Chemotaxis_Diffusion,9.654128,True
3,Chemotaxis,15735.0,Chemotaxis_Diffusion,9.663643,True
4,Chemotaxis,15363.0,Chemotaxis_Diffusion,9.639717,True


In [21]:

# Prepare the model formula
formula = 'Log_Time ~ Chemotaxis'

# Fit the OLS regression model
model_chemotaxis = sm.OLS.from_formula(formula, data=chemotaxis_df).fit()

# Display the summary of the model
model_chemotaxis_summary = model_chemotaxis.summary()
print(model_chemotaxis_summary)

                            OLS Regression Results                            
Dep. Variable:               Log_Time   R-squared:                       0.454
Model:                            OLS   Adj. R-squared:                  0.451
Method:                 Least Squares   F-statistic:                     161.3
Date:                Mon, 22 Jul 2024   Prob (F-statistic):           2.71e-27
Time:                        16:41:03   Log-Likelihood:                 187.69
No. Observations:                 196   AIC:                            -371.4
Df Residuals:                     194   BIC:                            -364.8
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              9.9068      0

### Mixed-Effects Model

In [5]:
# Step 1: Separate Regression Models

# Filter data for Chemotaxis_Diffusion
chemotaxis_df = combined_df[combined_df['Dataset'] == 'Chemotaxis_Diffusion'].copy()
# Fit OLS regression model for Chemotaxis
model_chemotaxis = sm.OLS.from_formula('Log_Time ~ Chemotaxis', data=chemotaxis_df).fit()
model_chemotaxis_summary = model_chemotaxis.summary()
print(model_chemotaxis_summary)

# Filter data for Conjugation_Ra
conjugation_ra_df = combined_df[combined_df['Dataset'] == 'Conjugation_Ra'].copy()
# Create the new column with opposite boolean values
conjugation_ra_df["No_R_a"] = ~conjugation_ra_df["R_a"]
# Add interaction term for No_R_a * Conjugation
conjugation_ra_df['No_R_a_Conj'] = conjugation_ra_df['No_R_a'] & conjugation_ra_df['Conjugation']
# Fit OLS regression model for R_a_Conj
model_ra_conj = sm.OLS.from_formula('Log_Time ~ Conjugation + No_R_a_Conj', data=conjugation_ra_df).fit()
model_ra_conj_summary = model_ra_conj.summary()
print(model_ra_conj_summary)

# Filter data for SOS_Spontaneous
sos_spontaneous_df = combined_df[combined_df['Dataset'] == 'SOS_Spontaneous'].copy()
# Fit OLS regression model for R_a_Conj
model_sos_spontaneous = sm.OLS.from_formula('Log_Time ~ SOS', data=sos_spontaneous_df).fit()
model_sos_spontaneous_summary = model_sos_spontaneous.summary()
print(model_sos_spontaneous_summary)

                            OLS Regression Results                            
Dep. Variable:               Log_Time   R-squared:                       0.454
Model:                            OLS   Adj. R-squared:                  0.451
Method:                 Least Squares   F-statistic:                     161.3
Date:                Mon, 22 Jul 2024   Prob (F-statistic):           2.71e-27
Time:                        17:35:54   Log-Likelihood:                 187.69
No. Observations:                 196   AIC:                            -371.4
Df Residuals:                     194   BIC:                            -364.8
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              9.9068      0

ValueError: zero-size array to reduction operation maximum which has no identity

In [10]:
# Step 2: Mixed-Effects Model
# Combine both datasets for mixed-effects model
combined_df_mixed = pd.concat([chemotaxis_df, conjugation_ra_df, sos_spontaneous_df], ignore_index=True)
combined_df_mixed.loc[combined_df_mixed['Dataset'] == 'Conjugation_Ra', 'Chemotaxis']
# Set 'Chemotaxis' values to NaN for the 'Conjugation_Ra' dataset
combined_df_mixed['Chemotaxis'] = combined_df_mixed['Chemotaxis'].astype(bool)
# combined_df_mixed.loc[combined_df_mixed['Dataset'] == 'Conjugation_Ra', 'Chemotaxis'] = pd.NA
combined_df_mixed['No_R_a'] = combined_df_mixed['No_R_a'].astype(bool)
# combined_df_mixed.loc[combined_df_mixed['Dataset'] == 'Conjugation_Ra', 'Chemotaxis'] = pd.NA
# Add interaction term for No_R_a * Conjugation
combined_df_mixed['No_R_a_Conj'] = combined_df_mixed['No_R_a'] & combined_df_mixed['Conjugation']


# Verify the changes
combined_df_mixed.info()
combined_df_mixed.tail()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 588 entries, 0 to 587
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Condition    588 non-null    object 
 1   Time         588 non-null    float64
 2   Dataset      588 non-null    object 
 3   Log_Time     588 non-null    float64
 4   Conjugation  588 non-null    bool   
 5   R_a          588 non-null    bool   
 6   Chemotaxis   588 non-null    bool   
 7   SOS          588 non-null    bool   
 8   No_R_a       588 non-null    bool   
 9   No_R_a_Conj  588 non-null    bool   
dtypes: bool(6), float64(2), object(2)
memory usage: 21.9+ KB


,Condition,Time,Dataset,Log_Time,Conjugation,R_a,Chemotaxis,SOS,No_R_a,No_R_a_Conj
583,No R_a No Conj,7081.0,Conjugation_Ra,8.865170,False,False,True,True,True,False
584,No R_a No Conj,7508.0,Conjugation_Ra,8.923724,False,False,True,True,True,False
585,No R_a No Conj,7013.0,Conjugation_Ra,8.855521,False,False,True,True,True,False
586,No R_a No Conj,4643.0,Conjugation_Ra,8.443116,False,False,True,True,True,False
587,No R_a No Conj,5652.0,Conjugation_Ra,8.639765,False,False,True,True,True,False


In [11]:
# Prepare the mixed-effects model formula
mixed_formula = 'Log_Time ~ Chemotaxis + Conjugation + No_R_a_Conj'

# Fit the mixed-effects model
mixed_model = MixedLM.from_formula(mixed_formula, groups='Dataset', data=combined_df_mixed, missing='drop').fit()

# Display the summary of the mixed-effects model
mixed_model_summary = mixed_model.summary()
print(mixed_model_summary)

            Mixed Linear Model Regression Results
Model:               MixedLM   Dependent Variable:   Log_Time 
No. Observations:    588       Method:               REML     
No. Groups:          2         Scale:                0.1105   
Min. group size:     196       Log-Likelihood:       -198.7945
Max. group size:     392       Converged:            Yes      
Mean group size:     294.0                                    
--------------------------------------------------------------
                    Coef.  Std.Err.   z    P>|z| [0.025 0.975]
--------------------------------------------------------------
Intercept            9.470    0.811 11.678 0.000  7.881 11.059
Chemotaxis[T.True]  -0.170    0.047 -3.581 0.000 -0.263 -0.077
Conjugation[T.True] -0.166    0.041 -4.010 0.000 -0.247 -0.085
No_R_a_Conj[T.True] -0.210    0.047 -4.430 0.000 -0.304 -0.117
Dataset Var          1.311    5.530                           

